# EWS 2D MAE pretraining (standalone from repo `pretrain/`)

Same **text list → load by path** style as AR-SSL4M; does **not** modify files under root `pretrain/`.

## Data

- **`float32` `.npy`** crops, shape **`(350, 350, 3)`**, under **`EWS/data/npy/`**（ `NPY_DIR`； `D:\Cursor\UNSW-COMP-9517\EWS\data\npy`）。
- Cell below builds **`EWS/data/pretrain/patch_lists/all_patches.txt`** (one absolute path per line).

## Model

- **2D MAE**: `Conv2d` patch embed + `TransformerEncoder`, **RGB**, **`patch_size=5`** (350/5 = 70×70 patches)。 **`batch_size=1`**。
- Code in **`EWS/pretrain/`** (`dataset_npy.py`, `mae2d.py`, `train_mae2d.py`).

## Dependencies

- `torch` (**CUDA build**; **RTX 50 / Blackwell needs cu128-class wheels**, cu124 may  `no kernel image`), `tqdm`.
- Training **defaults to GPU**; pass `device="cpu"` only for debugging.


In [ ]:
from pathlib import Path


def find_repo(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "EWS" / "pretrain" / "train_mae2d.py").is_file():
            return p
    return start.resolve()


REPO = find_repo(Path.cwd())
NPY_DIR = REPO / "EWS" / "data" / "npy"
_NPY_FALLBACK = Path(r"D:\Cursor\UNSW-COMP-9517\EWS\data\npy")
if not NPY_DIR.is_dir() and _NPY_FALLBACK.is_dir():
    NPY_DIR = _NPY_FALLBACK
LIST_DIR = REPO / "EWS" / "data" / "pretrain" / "patch_lists"
LIST_FILE = LIST_DIR / "all_patches.txt"
CKPT_DIR = REPO / "EWS" / "data" / "pretrain" / "checkpoints"

if not NPY_DIR.is_dir():
    NPY_DIR = REPO / "EWS" / "data"

print("NPY_DIR:", NPY_DIR.resolve())

LIST_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
count_1 = 0
count_2 = 0

paths = sorted(NPY_DIR.glob("*.npy"))
with open(LIST_FILE, "w", encoding="utf-8") as f:
    for p in paths:
        f.write(str(p.resolve()) + "\n")
        count_1 += 1
        count_2 += 1
        if count_2 >= 100:
            print(count_1)
            count_2 = 0

print("list_path:", LIST_FILE)
print("num_samples:", len(paths))


In [ ]:
# Blackwell (RTX 5090, sm_120): need CUDA 12.8+ wheels; cu124 may hit no kernel image.
# Default: nightly cu128 + --pre. Override with TORCH_INDEX_URL / TORCH_PIP_PRE=0; see install.py.
import os
import subprocess
import sys

idx = os.environ.get("TORCH_INDEX_URL", "https://download.pytorch.org/whl/nightly/cu128")
extra = ["--pre"] if os.environ.get("TORCH_PIP_PRE", "1") not in ("0", "false", "False") else []
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", *extra, "torch", "--index-url", idx]
)


In [ ]:
import torch

v = torch.__version__
print("torch:", v)
print("torch.version.cuda:", torch.version.cuda)

if "+cpu" in v:
    print("CPU-only torch detected")

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("compute capability:", cap)
    print("GPU:", torch.cuda.get_device_name(0))
    a = torch.randn(256, 256, device="cuda")
    b = torch.randn(256, 256, device="cuda")
    _ = a @ b
    torch.cuda.synchronize()
else:
    print("CUDA not available")


In [ ]:
import sys

EWS_PRE = REPO / "EWS" / "pretrain"
sys.path.insert(0, str(EWS_PRE))

from dataset_npy import NpyPatchListDataset

ds = NpyPatchListDataset(LIST_FILE, split="train")
print("train:", len(ds), "val:", len(NpyPatchListDataset(LIST_FILE, split="val")))
x = ds[0]
print("single sample shape (3,H,W):", x.shape)


In [ ]:
import sys

sys.path.insert(0, str(REPO / "EWS" / "pretrain"))
from train_mae2d import run_training

run_training(
    list_path=str(LIST_FILE.resolve()),
    ckpt_dir=str(CKPT_DIR.resolve()),
    img_size=350,
    patch_size=25,
    epochs=1,
    batch_size=1,
    num_workers=2,
    val_every=500,
    mask_ratio=0.75,
)
